# Migration Scope POC (Real Project: cantor)
Notebook này chạy trực tiếp trên dữ liệu thật ở `freshbrew_data/cantor` để xác định vùng code Java cần ưu tiên upgrade.
Luồng chính:
1. Khởi tạo cấu hình thật: `REAL_PROJECT_DIR`, `JDK_TARGET`, `LIB_TARGET`.
2. Parse AST từng file Java và tìm method bị ảnh hưởng bởi legacy/import pattern.
3. Xuất JSON report ROI toàn dự án.
4. Visualize timeline + highlight block code để reviewer/dev xử lý nhanh.

## Thiết lập (Setup)

Mục đích: mô tả các biến cấu hình và cách kết nối kernel (môi trường ảo) để chạy notebook.

Gồm:
- `PROJECT_ROOT`, `REAL_PROJECT_PATH`, `POM_PATH`
- Thiết lập `JDK_TARGET` và `LIB_TARGET` (tự động trích xuất từ `pom.xml` khi có)
- Yêu cầu: `tree-sitter`, `python` kernel tương ứng, (tùy chọn) `z3-solver` nếu cần thực thi bước solver.

In [6]:
from pathlib import Path
import json
import os
import sys

from src.tools.maven_upgrade_tools import index_java_project, run_upgrade_pipeline

WORKSPACE_DIR = Path.cwd()
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.append(str(WORKSPACE_DIR))

REAL_PROJECT_DIR = Path(r"D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\cantor")
JDK_TARGET = "17"
OUTPUT_JSON = WORKSPACE_DIR / "affected_scopes_cantor.json"

index_result = index_java_project(str(REAL_PROJECT_DIR))
if index_result.get("status") != "ok":
    raise RuntimeError(f"Cannot index project: {index_result}")

real_dependencies = index_result.get("dependencies", [])
current_libs = {
    f"{d.get('groupId')}:{d.get('artifactId')}": d.get("version")
    for d in real_dependencies
    if d.get("groupId") and d.get("artifactId")
}

pipeline_result = run_upgrade_pipeline(real_dependencies, target_java=JDK_TARGET, max_versions=3, max_solutions=3)
solver_solutions = pipeline_result.get("solutions", [])
if solver_solutions and isinstance(solver_solutions[0], dict):
    LIB_TARGET = solver_solutions[0]
else:
    LIB_TARGET = current_libs

print(f"[OK] REAL_PROJECT_DIR: {REAL_PROJECT_DIR}")
print(f"[OK] JDK_TARGET: {JDK_TARGET}")
print(f"[OK] Dependencies: {len(current_libs)}")
print(f"[OK] LIB_TARGET entries: {len(LIB_TARGET)}")
for i, (k, v) in enumerate(LIB_TARGET.items(), start=1):
    if i > 10:
        break
    print(f"  - {k} -> {v}")

[OK] REAL_PROJECT_DIR: D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\cantor
[OK] JDK_TARGET: 17
[OK] Dependencies: 3
[OK] LIB_TARGET entries: 3
  - junit:junit -> 4.13.1
  - org.slf4j:slf4j-api -> 1.7.36
  - org.apache.hadoop:hadoop-common -> 3.5.0


## `MigrationScopeExtractor` — ý tưởng và hoạt động

Giải thích ngắn:
- Dùng `tree-sitter` (qua `UniversalASTReader`) để parse file Java.
- Query không chỉ bắt `method_declaration`, mà còn mở rộng sang `field_declaration` và `import_declaration`.
- Mục tiêu là bắt được cả import/package reference lẫn khai báo field static như `Logger`.
- Trích xuất node cùng với 5 dòng ngữ cảnh trước/sau để review.

Input: danh sách file nguồn (Java) và `LIB_TARGET`.
Output: danh sách ROI dưới dạng JSON (file, start_line, end_line, capture_name, context, reason).

In [7]:
import uuid
import re

try:
    from src.core.tree_sitter_engine import UniversalASTReader
except Exception as exc:
    UniversalASTReader = None
    print(f"[WARN] Could not import UniversalASTReader: {exc}")

class MigrationScopeExtractor:
    def __init__(self, file_path, legacy_pkgs):
        self.file_path = Path(file_path)
        self.legacy_packages = list(legacy_pkgs)
        self.source_text = self.file_path.read_text(encoding="utf-8", errors="replace")
        self.source_bytes = self.source_text.encode("utf-8")

        if UniversalASTReader is None:
            raise RuntimeError(
                "UniversalASTReader is unavailable. Check tree-sitter dependencies and import path."
            )

        self.ast_reader = UniversalASTReader()
        self.tree = self.ast_reader.parse(self.source_bytes, "java")

    def _package_pattern(self, package_name: str) -> re.Pattern:
        # Match either the full package/class string or a word-boundary token form.
        escaped = re.escape(package_name)
        token = re.escape(package_name.split(".")[-1])
        return re.compile(rf"(?<!\w)({escaped}|{token})(?!\w)")

    def find_affected_scopes(self):
        query = """
        (method_declaration) @method
        (field_declaration) @field
        (import_declaration) @import
        """
        matched_nodes = self.ast_reader.query(self.tree, "java", query)
        patterns = {pkg: self._package_pattern(pkg) for pkg in self.legacy_packages}
        results = []
        seen_ranges = set()

        for node, capture_name in matched_nodes:
            if capture_name not in {"method", "field", "import"}:
                continue

            start_byte = node.start_byte
            end_byte = node.end_byte
            if (start_byte, end_byte) in seen_ranges:
                continue
            seen_ranges.add((start_byte, end_byte))

            node_text = self.source_bytes[start_byte:end_byte].decode("utf-8", errors="replace")
            legacy_hits = [pkg for pkg, pattern in patterns.items() if pattern.search(node_text)]

            if legacy_hits:
                results.append({
                    "scope_id": f"scope-{uuid.uuid4().hex[:8]}",
                    "capture_name": capture_name,
                    "start_byte": start_byte,
                    "end_byte": end_byte,
                    "legacy_hits": legacy_hits,
                    "code_snippet": node_text,
                })

        return results

print("[OK] MigrationScopeExtractor ready for real project scan.")

[OK] MigrationScopeExtractor ready for real project scan.


## Chạy trên dự án thực (`real-project`)

Mục đích: thực thi pipeline scan trên `freshbrew_data/cantor` để tự động xác định `JDK_TARGET` và `LIB_TARGET`, rồi scan toàn bộ source Java.

Ghi chú vận hành:
- Tập tin `pom.xml` được đọc để ước lượng phiên bản JDK và phiên bản thư viện hiện tại.
- Bộ pattern scan được mở rộng từ groupId/artifactId thành các hint thực tế như `org.slf4j`, `org.apache.hadoop`, `LoggerFactory`, `Logger`, `Writable`.
- Kết quả sơ bộ lưu vào `affected_scopes_cantor.json` và `affected_scopes_cantor.csv`.

In [8]:
# Build scan patterns from target libs + common legacy Java EE tokens

def dependency_hints(library_name):
    group_id, artifact_id = library_name.split(":", 1)
    hints = {group_id, artifact_id}
    group_tail = group_id.split(".")[-1]
    hints.add(group_tail)

    if group_id == "org.slf4j":
        hints.update({"org.slf4j", "slf4j", "LoggerFactory", "Logger"})
    elif group_id == "org.apache.hadoop":
        hints.update({"org.apache.hadoop", "hadoop", "Writable"})
    elif artifact_id.endswith("-api"):
        hints.add(artifact_id[:-4])

    return {hint for hint in hints if hint}

SCAN_PATTERNS = sorted({
    *[hint for lib in LIB_TARGET.keys() if isinstance(lib, str) for hint in dependency_hints(lib)],
    "javax.xml.bind",
    "javax.activation",
    "javax.annotation",
    "org.joda.time",
})

java_files = []
for root, dirs, files in os.walk(REAL_PROJECT_DIR):
    dirs[:] = [d for d in dirs if d not in {".git", "target", "build", ".idea", ".vscode"}]
    for name in files:
        if name.endswith(".java"):
            java_files.append(Path(root) / name)

project_affected_scopes = []
for file_path in java_files:
    try:
        extractor = MigrationScopeExtractor(file_path, SCAN_PATTERNS)
        scopes = extractor.find_affected_scopes()
        for scope in scopes:
            scope["file_path"] = str(file_path.relative_to(REAL_PROJECT_DIR))
        project_affected_scopes.extend(scopes)
    except Exception as exc:
        print(f"[WARN] Skip {file_path}: {exc}")

OUTPUT_JSON.write_text(json.dumps(project_affected_scopes, ensure_ascii=False, indent=2), encoding="utf-8")
affected_scopes = project_affected_scopes

print(f"Java files scanned: {len(java_files)}")
print(f"Total affected scopes: {len(affected_scopes)}")
print(f"[OK] Report written to: {OUTPUT_JSON}")
for i, scope in enumerate(affected_scopes[:10], start=1):
    print("-" * 72)
    print(f"ROI #{i} | file={scope.get('file_path')}")
    print(f"capture={scope.get('capture_name')} | bytes={scope['start_byte']}..{scope['end_byte']} | hits={scope['legacy_hits']}")

Java files scanned: 5
Total affected scopes: 8
[OK] Report written to: d:\capstone_project\MYGRATE---Capstone-Project\affected_scopes_cantor.json
------------------------------------------------------------------------
ROI #1 | file=src\main\java\com\adroll\cantor\HLLWritable.java
capture=import | bytes=161..198 | hits=['Writable', 'hadoop', 'org.apache.hadoop']
------------------------------------------------------------------------
ROI #2 | file=src\main\java\com\adroll\cantor\HLLWritable.java
capture=import | bytes=199..223 | hits=['Logger', 'org.slf4j', 'slf4j']
------------------------------------------------------------------------
ROI #3 | file=src\main\java\com\adroll\cantor\HLLWritable.java
capture=import | bytes=224..255 | hits=['LoggerFactory', 'org.slf4j', 'slf4j']
------------------------------------------------------------------------
ROI #4 | file=src\main\java\com\adroll\cantor\HLLWritable.java
capture=field | bytes=488..565 | hits=['Logger', 'LoggerFactory']
----------

In [9]:
# Text summary only (no visualization)
from collections import Counter

if not affected_scopes:
    print("No affected scopes found.")
else:
    by_file = Counter(scope.get("file_path", "unknown") for scope in affected_scopes)
    hit_counter = Counter()
    for scope in affected_scopes:
        for hit in scope.get("legacy_hits", []):
            hit_counter[hit] += 1

    print("Affected files:")
    for file_path, count in by_file.most_common(20):
        print(f"  - {file_path}: {count} scope(s)")

    print("\nTop legacy hits:")
    for name, cnt in hit_counter.most_common(20):
        print(f"  - {name}: {cnt}")

Affected files:
  - src\main\java\com\adroll\cantor\HLLWritable.java: 4 scope(s)
  - src\test\java\com\adroll\cantor\TestHLLCounter.java: 2 scope(s)
  - src\test\java\com\adroll\cantor\TestHLLWritable.java: 2 scope(s)

Top legacy hits:
  - junit: 4
  - Logger: 2
  - org.slf4j: 2
  - slf4j: 2
  - LoggerFactory: 2
  - Writable: 1
  - hadoop: 1
  - org.apache.hadoop: 1


In [10]:
# Print local upgrade context in GitHub-like style
from bisect import bisect_right

CONTEXT_LINES = 5

if not affected_scopes:
    print("No ROI to print.")
else:
    scopes_by_file = {}
    for scope in affected_scopes:
        file_rel = scope.get("file_path")
        if not file_rel:
            continue
        scopes_by_file.setdefault(file_rel, []).append(scope)

    for file_rel in sorted(scopes_by_file.keys()):
        abs_file = REAL_PROJECT_DIR / file_rel
        src_text = abs_file.read_text(encoding="utf-8", errors="replace")
        lines = src_text.splitlines()

        # Build UTF-8 byte -> char index map for accurate byte-to-line conversion
        byte_marks = [0]
        char_marks = [0]
        running = 0
        for idx, ch in enumerate(src_text, start=1):
            running += len(ch.encode("utf-8"))
            byte_marks.append(running)
            char_marks.append(idx)

        def byte_to_char_idx(byte_offset: int) -> int:
            pos = bisect_right(byte_marks, max(0, byte_offset)) - 1
            pos = max(0, min(pos, len(char_marks) - 1))
            return char_marks[pos]

        def char_to_line(char_idx: int) -> int:
            return src_text.count("\n", 0, max(0, char_idx)) + 1

        print("=" * 100)
        print(f"File: {file_rel}")
        print("=" * 100)

        sorted_scopes = sorted(scopes_by_file[file_rel], key=lambda s: s["start_byte"] )
        for i, scope in enumerate(sorted_scopes, start=1):
            s_line = char_to_line(byte_to_char_idx(scope["start_byte"]))
            e_line = char_to_line(byte_to_char_idx(scope["end_byte"]))

            pre_start = max(1, s_line - CONTEXT_LINES)
            post_end = min(len(lines), e_line + CONTEXT_LINES)

            print("-" * 100)
            print(
                f"ROI #{i} | lines {s_line}-{e_line} | bytes {scope['start_byte']}-{scope['end_byte']} | "
                f"hits={scope['legacy_hits']}"
            )
            print("-" * 100)

            # before
            for ln in range(pre_start, s_line):
                print(f"{ln:5d}   {lines[ln - 1]}")

            # block
            for ln in range(s_line, e_line + 1):
                print(f"{ln:5d} > {lines[ln - 1]}")

            # after
            for ln in range(e_line + 1, post_end + 1):
                print(f"{ln:5d}   {lines[ln - 1]}")

            print()

File: src\main\java\com\adroll\cantor\HLLWritable.java
----------------------------------------------------------------------------------------------------
ROI #1 | lines 9-9 | bytes 161-198 | hits=['Writable', 'hadoop', 'org.apache.hadoop']
----------------------------------------------------------------------------------------------------
    4   import java.io.DataOutput;
    5   import java.io.IOException;
    6   import java.util.Arrays;
    7   import java.util.TreeSet;
    8   
    9 > import org.apache.hadoop.io.Writable;
   10   import org.slf4j.Logger;
   11   import org.slf4j.LoggerFactory;
   12   
   13   import com.adroll.cantor.HLLCounter;
   14   

----------------------------------------------------------------------------------------------------
ROI #2 | lines 10-10 | bytes 199-223 | hits=['Logger', 'org.slf4j', 'slf4j']
----------------------------------------------------------------------------------------------------
    5   import java.io.IOException;
    6   impo

In [11]:
# Export a compact upgrade planning table (no external deps)
import csv

rows = []
for scope in affected_scopes:
    rows.append({
        "file_path": scope.get("file_path", ""),
        "scope_id": scope.get("scope_id", ""),
        "start_byte": scope.get("start_byte", -1),
        "end_byte": scope.get("end_byte", -1),
        "legacy_hits": ", ".join(scope.get("legacy_hits", [])),
    })

ROI_CSV = WORKSPACE_DIR / "affected_scopes_cantor.csv"
fieldnames = ["file_path", "scope_id", "start_byte", "end_byte", "legacy_hits"]

with open(ROI_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in sorted(rows, key=lambda r: (r["file_path"], r["start_byte"])):
        writer.writerow(row)

print(f"[OK] CSV written: {ROI_CSV}")
print("Preview:")
for row in sorted(rows, key=lambda r: (r["file_path"], r["start_byte"]))[:20]:
    print(
        f"  - {row['file_path']} | {row['scope_id']} | "
        f"{row['start_byte']}..{row['end_byte']} | {row['legacy_hits']}"
    )

[OK] CSV written: d:\capstone_project\MYGRATE---Capstone-Project\affected_scopes_cantor.csv
Preview:
  - src\main\java\com\adroll\cantor\HLLWritable.java | scope-3111a2a9 | 161..198 | Writable, hadoop, org.apache.hadoop
  - src\main\java\com\adroll\cantor\HLLWritable.java | scope-ad40f3e3 | 199..223 | Logger, org.slf4j, slf4j
  - src\main\java\com\adroll\cantor\HLLWritable.java | scope-2f07fdd7 | 224..255 | LoggerFactory, org.slf4j, slf4j
  - src\main\java\com\adroll\cantor\HLLWritable.java | scope-6e74b865 | 488..565 | Logger, LoggerFactory
  - src\test\java\com\adroll\cantor\TestHLLCounter.java | scope-b23f3aa5 | 237..270 | junit
  - src\test\java\com\adroll\cantor\TestHLLCounter.java | scope-4962f5c7 | 272..294 | junit
  - src\test\java\com\adroll\cantor\TestHLLWritable.java | scope-c3870e43 | 28..61 | junit
  - src\test\java\com\adroll\cantor\TestHLLWritable.java | scope-2fed8edd | 204..226 | junit


## Kết quả, artifacts và bước tiếp theo

Artifacts tạo ra:
- `affected_scopes_cantor.json` / `.csv`
- `dependency_focus_scopes.json`
- `artifacts/pom_line_mappings.json`

Gợi ý bước tiếp theo:
- Dùng `dependency_focus_scopes.json` để soạn các thay đổi mã (manual hoặc bằng `TranslatorAgent`).
- Viết patch cho `pom.xml` theo `pom_line_mappings.json` và chạy `mvn -DskipTests package` để kiểm tra biên dịch.
- Chạy smoke tests / runtime checks nếu có.

Nếu muốn, tôi có thể:
- Chạy toàn bộ notebook trên `freshbrew_data/cantor` ngay bây giờ
- Hoặc tạo patch tự động mẫu cho `pom.xml` và PR mẫu

In [12]:
# Final workflow recap (text-only output)
print("WORKFLOW (Real Project Mode)")
print("1) Load real project + infer LIB_TARGET and JDK_TARGET")
print("2) Build scan patterns from target libs + common legacy tokens")
print("3) Parse every Java file AST and extract affected methods")
print("4) Save JSON/CSV reports for translator or patch pipeline")
print("5) Print local code contexts for each ROI (GitHub-like, with line numbers)")

print()
print(f"REAL_PROJECT_DIR = {REAL_PROJECT_DIR}")
print(f"JDK_TARGET = {JDK_TARGET}")
print(f"LIB_TARGET entries = {len(LIB_TARGET)}")
print(f"Total ROI = {len(affected_scopes)}")
print(f"JSON report = {OUTPUT_JSON}")
print(f"CSV report = {WORKSPACE_DIR / 'affected_scopes_cantor.csv'}")

WORKFLOW (Real Project Mode)
1) Load real project + infer LIB_TARGET and JDK_TARGET
2) Build scan patterns from target libs + common legacy tokens
3) Parse every Java file AST and extract affected methods
4) Save JSON/CSV reports for translator or patch pipeline
5) Print local code contexts for each ROI (GitHub-like, with line numbers)

REAL_PROJECT_DIR = D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\cantor
JDK_TARGET = 17
LIB_TARGET entries = 3
Total ROI = 8
JSON report = d:\capstone_project\MYGRATE---Capstone-Project\affected_scopes_cantor.json
CSV report = d:\capstone_project\MYGRATE---Capstone-Project\affected_scopes_cantor.csv


## Ánh xạ `pom.xml` từ báo cáo solver

Mục tiêu: từ báo cáo của solver/architect, tìm chính xác dòng cần thay đổi trong `pom.xml` (vd. property hoặc `<version>` của dependency).

Output: in-file context (5 dòng trước/sau) và số dòng để reviewer dễ apply change.

Tệp sinh ra: `artifacts/pom_line_mappings.json` (mô tả dependency -> line range).

In [13]:
# Dependency line mapping from a mock final report
from pathlib import Path
import json

MOCK_FINAL_REPORT = {
    "status": "ok",
    "report_path": r"artifacts\codebase_discovery\cantor_upgrade_report.json",
    "summary": {
        "project": "cantor",
        "target_java": "17",
        "solver": "z3",
        "solutions": 5,
        "smoke_tests": 3,
    },
    "current_dependencies": [
        {"library": "junit:junit", "current_version": "4.11", "scope": "test", "target_version": None},
        {"library": "org.slf4j:slf4j-api", "current_version": "1.7.5", "scope": "compile", "target_version": "1.7.36"},
        {"library": "org.apache.hadoop:hadoop-common", "current_version": "2.2.0", "scope": "compile", "target_version": "3.5.0"},
    ],
    "selected_solution": {
        "org.slf4j:slf4j-api": "1.7.36",
        "org.apache.hadoop:hadoop-common": "3.5.0",
    },
}

POM_PATH = REAL_PROJECT_DIR / "pom.xml"

def build_line_index(text: str):
    lines = text.splitlines()
    return lines

def find_property_line(lines, property_name: str):
    needle = f"<{property_name}>"
    for idx, line in enumerate(lines, start=1):
        if needle in line:
            return idx
    return None

def find_dependency_block(lines, group_id: str, artifact_id: str):
    start_idx = None
    end_idx = None
    for idx, line in enumerate(lines, start=1):
        if f"<groupId>{group_id}</groupId>" in line and any(f"<artifactId>{artifact_id}</artifactId>" in l for l in lines[idx - 1: idx + 5]):
            start_idx = idx
            for j in range(idx, min(len(lines), idx + 20) + 1):
                if "</dependency>" in lines[j - 1]:
                    end_idx = j
                    break
            break
    return start_idx, end_idx

def dependency_version_line(lines, block_start, block_end, target_version_ref=None):
    if block_start is None or block_end is None:
        return None
    for idx in range(block_start, block_end + 1):
        if "<version>" in lines[idx - 1]:
            if target_version_ref is None or target_version_ref in lines[idx - 1]:
                return idx
    return None

pom_text = POM_PATH.read_text(encoding="utf-8")
pom_lines = build_line_index(pom_text)

resolved_targets = [
    item for item in MOCK_FINAL_REPORT["current_dependencies"]
    if item.get("target_version") is not None
 ]

line_map = []
for dep in resolved_targets:
    group_id, artifact_id = dep["library"].split(":", 1)
    block_start, block_end = find_dependency_block(pom_lines, group_id, artifact_id)
    if group_id == "org.apache.hadoop" and artifact_id == "hadoop-common":
        # This dependency is version-managed by <hadoop.version> in <properties>.
        version_line = find_property_line(pom_lines, "hadoop.version")
        change_kind = "property"
        change_target = "<hadoop.version>..."
        if version_line is None:
            version_line = dependency_version_line(pom_lines, block_start, block_end, "${hadoop.version}")
            change_kind = "dependency"
            change_target = "<version>${hadoop.version}</version>"
    else:
        version_line = dependency_version_line(pom_lines, block_start, block_end, dep["current_version"])
        change_kind = "dependency"
        change_target = f"<version>{dep['current_version']}</version>"

    line_map.append({
        "library": dep["library"],
        "current_version": dep["current_version"],
        "target_version": dep["target_version"],
        "scope": dep["scope"],
        "change_kind": change_kind,
        "line": version_line,
        "block_start": block_start,
        "block_end": block_end,
        "change_target": change_target,
    })

print("MOCK FINAL REPORT")
print(json.dumps(MOCK_FINAL_REPORT, indent=2, ensure_ascii=False))
print()
print("LINES TO CHANGE")
for item in line_map:
    print("-" * 100)
    print(f"Library: {item['library']}")
    print(f"Current -> Target: {item['current_version']} -> {item['target_version']}")
    print(f"Change kind: {item['change_kind']}")
    print(f"Pom line: {item['line']}")
    print(f"Block: {item['block_start']}..{item['block_end']}")

    if item["line"] is None:
        print("[WARN] Could not resolve exact line in pom.xml")
        continue

    context_start = max(1, item["line"] - 5)
    context_end = min(len(pom_lines), item["line"] + 5)
    print(f"File: {POM_PATH.relative_to(REAL_PROJECT_DIR)}")
    print(f"Context: lines {context_start}-{context_end}")
    for ln in range(context_start, context_end + 1):
        prefix = ">" if ln == item["line"] else " "
        print(f"{ln:5d} {prefix} {pom_lines[ln - 1]}")
    print()
    print(f"Suggested change target: {item['change_target']} -> {item['target_version']}")
    print()

MOCK FINAL REPORT
{
  "status": "ok",
  "report_path": "artifacts\\codebase_discovery\\cantor_upgrade_report.json",
  "summary": {
    "project": "cantor",
    "target_java": "17",
    "solver": "z3",
    "solutions": 5,
    "smoke_tests": 3
  },
  "current_dependencies": [
    {
      "library": "junit:junit",
      "current_version": "4.11",
      "scope": "test",
      "target_version": null
    },
    {
      "library": "org.slf4j:slf4j-api",
      "current_version": "1.7.5",
      "scope": "compile",
      "target_version": "1.7.36"
    },
    {
      "library": "org.apache.hadoop:hadoop-common",
      "current_version": "2.2.0",
      "scope": "compile",
      "target_version": "3.5.0"
    }
  ],
  "selected_solution": {
    "org.slf4j:slf4j-api": "1.7.36",
    "org.apache.hadoop:hadoop-common": "3.5.0"
  }
}

LINES TO CHANGE
----------------------------------------------------------------------------------------------------
Library: org.slf4j:slf4j-api
Current -> Target: 1.7.5 -

## Focus phụ thuộc (Dependency Focus) — tìm mã sử dụng thư viện mục tiêu

Mục đích: từ `LIB_TARGET` tạo báo cáo `dependency_focus_scopes.json` liệt kê các file và dòng có `import` hoặc `identifier` liên quan đến thư viện.

Đầu ra: JSON gồm `file`, `line`, `match_type` (`import`|`identifier`|`logger_call`), và ngữ cảnh 5 dòng.

In [14]:
# Dependency usage focus map: find source-code regions that use the selected libraries
import os
import re
import json
from collections import defaultdict

FOCUS_DEPENDENCIES = {
    "org.slf4j:slf4j-api": {
        "current_version": "1.7.5",
        "target_version": "1.7.36",
        "patterns": [r"org\.slf4j", r"LoggerFactory", r"\bLogger\b"],
    },
    "org.apache.hadoop:hadoop-common": {
        "current_version": "2.2.0",
        "target_version": "3.5.0",
        "patterns": [r"org\.apache\.hadoop"],
    },
}

CONTEXT = 5
focus_report = []

def cluster_lines(lines):
    if not lines:
        return []
    lines = sorted(set(lines))
    clusters = [[lines[0]]]
    for line_no in lines[1:]:
        if line_no - clusters[-1][-1] <= CONTEXT * 2 + 1:
            clusters[-1].append(line_no)
        else:
            clusters.append([line_no])
    return clusters

java_files = []
for root, dirs, files in os.walk(REAL_PROJECT_DIR):
    dirs[:] = [d for d in dirs if d not in {".git", "target", "build", ".idea", ".vscode"}]
    for name in files:
        if name.endswith(".java"):
            java_files.append(Path(root) / name)

for dep_name, dep_info in FOCUS_DEPENDENCIES.items():
    patterns = [re.compile(p) for p in dep_info["patterns"]]
    for file_path in java_files:
        src_text = file_path.read_text(encoding="utf-8", errors="replace")
        lines = src_text.splitlines()
        hit_lines = []

        for idx, line in enumerate(lines, start=1):
            if any(pattern.search(line) for pattern in patterns):
                hit_lines.append(idx)

        if not hit_lines:
            continue

        for cluster in cluster_lines(hit_lines):
            start_line = max(1, cluster[0] - CONTEXT)
            end_line = min(len(lines), cluster[-1] + CONTEXT)
            snippet = []
            for ln in range(start_line, end_line + 1):
                marker = ">" if ln in cluster else " "
                snippet.append(f"{ln:5d} {marker} {lines[ln - 1]}")

            focus_report.append({
                "dependency": dep_name,
                "current_version": dep_info["current_version"],
                "target_version": dep_info["target_version"],
                "file_path": str(file_path.relative_to(REAL_PROJECT_DIR)),
                "start_line": start_line,
                "end_line": end_line,
                "hit_lines": cluster,
                "snippet": "\n".join(snippet),
            })

focus_report = sorted(focus_report, key=lambda x: (x["file_path"], x["dependency"], x["start_line"]))

DEPENDENCY_FOCUS_JSON = WORKSPACE_DIR / "dependency_focus_scopes.json"
DEPENDENCY_FOCUS_JSON.write_text(json.dumps(focus_report, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Source files scanned: {len(java_files)}")
print(f"Focus regions found: {len(focus_report)}")
print(f"[OK] Focus report written to: {DEPENDENCY_FOCUS_JSON}")
print()

current_file = None
for item in focus_report:
    if item["file_path"] != current_file:
        current_file = item["file_path"]
        print("=" * 100)
        print(f"File: {current_file}")
        print("=" * 100)

    print("-" * 100)
    print(f"Dependency: {item['dependency']}  |  {item['current_version']} -> {item['target_version']}")
    print(f"Line range: {item['start_line']}..{item['end_line']}")
    print(f"Hit lines: {item['hit_lines']}")
    print(item["snippet"])

    print()

Source files scanned: 5
Focus regions found: 2
[OK] Focus report written to: d:\capstone_project\MYGRATE---Capstone-Project\dependency_focus_scopes.json

File: src\main\java\com\adroll\cantor\HLLWritable.java
----------------------------------------------------------------------------------------------------
Dependency: org.apache.hadoop:hadoop-common  |  2.2.0 -> 3.5.0
Line range: 4..14
Hit lines: [9]
    4   import java.io.DataOutput;
    5   import java.io.IOException;
    6   import java.util.Arrays;
    7   import java.util.TreeSet;
    8   
    9 > import org.apache.hadoop.io.Writable;
   10   import org.slf4j.Logger;
   11   import org.slf4j.LoggerFactory;
   12   
   13   import com.adroll.cantor.HLLCounter;
   14   

----------------------------------------------------------------------------------------------------
Dependency: org.slf4j:slf4j-api  |  1.7.5 -> 1.7.36
Line range: 5..27
Hit lines: [10, 11, 22]
    5   import java.io.IOException;
    6   import java.util.Arrays;


## Translator smoke run

Cell này gọi trực tiếp `TranslatorAgent` trên hai artifact đã sinh từ phần scope discovery để tạo change plan có cấu trúc.


In [ ]:
from pathlib import Path
import json

from src.agents.translator_agent import TranslatorAgent

PROJECT_ROOT = Path(r"D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\cantor")
PAYLOAD = {
    "project_path": str(PROJECT_ROOT),
    "target_java_version": "17",
    "project_type": "java",
    "current_instruction": "Notebook translator smoke run",
    "migration_tasks": [
        {"title": "Review dependency scope changes"},
        {"title": "Prepare migration patch plan"},
    ],
    "dependency_focus_report_path": str(Path.cwd() / "dependency_focus_scopes.json"),
    "affected_scopes_path": str(Path.cwd() / "affected_scopes_cantor.json"),
}

agent = TranslatorAgent()
translator_report = json.loads(agent.run(json.dumps(PAYLOAD, ensure_ascii=False)))

print(f"[OK] status={translator_report.get('status')}")
print(f"[OK] task_count={translator_report.get('task_count')}")
print(f"[OK] candidate_count={translator_report.get('summary', {}).get('candidate_count')}")
print(f"[OK] files_covered={translator_report.get('summary', {}).get('files_covered')}")
for index, candidate in enumerate(translator_report.get('change_candidates', [])[:3], start=1):
    print("-" * 72)
    print(f"Candidate #{index}: {candidate.get('file_path')}")
    print(f"lines={candidate.get('start_line')}..{candidate.get('end_line')}")
    print(f"dependency={candidate.get('dependency')}")
